In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


['subtype_for_TCGA-KIRC.tsv',
 'samples_for_TCGA-STAD_Stomach_subtype_adenocarcinoma_generic_tumor_adenocarcinoma_subtype_tissue_adenocarcinoma_generic.tsv',
 'mutations_anal_for_study_TCGA-KIRC_Kidney_subtype_adenocarcinoma_generic_tumor_adenocarcinoma_subtype_tissue_adenocarcinoma_generic.tsv',
 'mutations_anal_for_study_TCGA-THCA_Thyroid_gland_subtype_other_tumor_other_subtype_tissue_other.tsv',
 'mutations_summ_for_study_TCGA-UCS_Uterus_NOS_Corpus_uteri_subtype_other_tumor_melanoma_subtype_tissue_other.tsv',
 'mutations_summ_for_study_TCGA-GBM_Brain_subtype_other_tumor_other_subtype_tissue_other.tsv',
 'mutations_anal_for_study_TCGA-LUAD_Bronchus_and_lung_subtype_other_tumor_other_subtype_tissue_other.tsv',
 'mutations_anal_for_study_TCGA-CHOL_Liver_and_intrahepatic_bile_ducts_Other_and_unspecified_parts_of_biliary_tract_Pancreas_subtype_sarcoma_tumor_sarcoma_subtype_tissue_sarcoma.tsv',
 'mutations_anal_for_study_TCGA-CHOL_Liver_and_intrahepatic_bile_ducts_Other_and_unspecified_pa

### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [20]:
def loop_program_psi_samples(program:str='TCGA', ipsi:Any=None, force:bool=False,
        verbose:bool=True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

    df_cases, df_subt, df_prof = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if isinstance(ipsi, int):
        lista = [ipsi]
    else:
        lista = np.arange(len(df_psi))

    df_list_cases, df_list_samples, df_list_mutations = [], [], []

    for ipsi in lista:
        row = df_psi.iloc[ipsi]
        pid = row.pid
        primary_site = row.primary_site

        print(f'{ipsi}) {primary_site}', end=' - ')

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

        if df_cases.empty:
            print("No cases found for PID:", pid)
            continue

        if isinstance(df_cases, pd.DataFrame):
            df_list_cases.append(df_cases)
        else:
            print("Unexpected type for df_cases:", type(df_cases))
            raise Exception("Stope: unexpected type for df_cases")


        for isubt, row in df_subt.iterrows():
            subtype_global = row.subtype_global
            tumor_class = row.tumor_class
            subtype_tissue = row.subtype_tissue

            s_case = f"{pid}_{primary_site}_subtype_{subtype_global}_tumor_{tumor_class}_subtype_tissue_{subtype_tissue}"

            if len(s_case) > 180:
                s_case = f"{pid}_{primary_site[:40]}_subtype_{subtype_global[:40]}_tumor_{tumor_class[:40]}_tissue_{subtype_tissue[:40]}"

            s_case = title_replace(s_case)

            print(f'{isubt}) {s_case}')

            df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                                          tumor_class=tumor_class, subtype_tissue=subtype_tissue, s_case=s_case,
                                                          batch_size=200, force=force, verbose=verbose)
            
            if df_samples.empty:
                print("No samples found for PID:", pid)
                continue

            df_list_samples.append(df_samples)

            df2 = df_samples[~df_samples.sample_type.str.contains('Blood', case=False, na=False)]

            if df2.empty:
                print("No samples having non-blood types for PID:", pid)
                continue

            barcodes = list(np.unique(df2.barcode_id))

            print("Getting mutations", end=' ')
            dff, _ = gdc.get_df_mut_transform_mutation_table(study_id=pid, s_case=s_case, sample_ids=barcodes, force=force, verbose=verbose)

            if dff.empty:
                print("Could not find mutations for :", s_case)
                continue

            df_list_mutations.append(dff)

    if len(df_list_cases) > 0:
        df_all_cases = pd.concat(df_list_cases, ignore_index=True)
        df_all_cases = df_all_cases.drop_duplicates()
        df_all_cases = df_all_cases.reset_index(drop=True)
    else:
        df_all_cases = pd.DataFrame()

    if len(df_list_samples) > 0:
        df_all_samples = pd.concat(df_list_samples, ignore_index=True)
        df_all_samples = df_all_samples.drop_duplicates()
        df_all_samples = df_all_samples.reset_index(drop=True)
    else:
        df_all_samples = pd.DataFrame()

    if len(df_list_mutations) > 0:
        df_all_mutations = pd.concat(df_list_mutations, ignore_index=True)
        df_all_mutations = df_all_mutations.drop_duplicates()
        df_all_mutations = df_all_mutations.reset_index(drop=True)
    else:
        df_all_mutations = pd.DataFrame()        

    return df_psi, df_all_cases, df_all_samples, df_all_mutations



In [ ]:
force=False
verbose=False

pid = 'TCGA'

df_psi, df_all_cases, df_all_samples, df_all_mutations = \
    loop_program_psi_samples(program=pid, ipsi=None, force=force, verbose=verbose)

print("\n----------- end ------------\n")
print(len(df_all_samples))


0) Adrenal gland - 0) TCGA-ACC_Adrenal_gland_subtype_adrenal_cortical_carcinoma_tumor_adrenal_cortical_carcinoma_subtype_tissue_adrenal_cortical_carcinoma
Getting mutations 
>>> acc_tcga_pan_can_atlas_2018 --> TCGA-ACC_Adrenal_gland_subtype_adrenal_cortical_carcinoma_tumor_adrenal_cortical_carcinoma_subtype_tissue_adrenal_cortical_carcinoma len = 54 - [np.str_('TCGA-OR-A5J2-01'), np.str_('TCGA-OR-A5J4-01'), np.str_('TCGA-OR-A5J5-01'), np.str_('TCGA-OR-A5J7-01'), np.str_('TCGA-OR-A5J8-01')]...
1) TCGA-ACC_Adrenal_gland_subtype_adrenal_cortical_carcinoma_tumor_adrenal_cortical_carcinoma_subtype_tissue_adrenal_cortical_carcinoma
Getting mutations 
>>> acc_tcga_pan_can_atlas_2018 --> TCGA-ACC_Adrenal_gland_subtype_adrenal_cortical_carcinoma_tumor_adrenal_cortical_carcinoma_subtype_tissue_adrenal_cortical_carcinoma len = 54 - [np.str_('TCGA-OR-A5J2-01'), np.str_('TCGA-OR-A5J4-01'), np.str_('TCGA-OR-A5J5-01'), np.str_('TCGA-OR-A5J7-01'), np.str_('TCGA-OR-A5J8-01')]...
2) TCGA-ACC_Adrenal_gla

In [22]:
print(len(df_all_cases))
df_all_cases.tail(3)

11428


,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac
11425,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms",db67a9cd-65ba-4d6f-a799-031b6b3df599,[{'primary_diagnosis': 'Serous cystadenocarcin...,TCGA-UCEC,serous,unknown,"Serous cystadenocarcinoma, NOS",G3,NaN,...,corpus uteri,cystic mucinous and serous neoplasms,serous cystadenocarcinoma,adenocarcinoma,epithelial,uterine_serous,ok,valid,1,0.001786
11426,Corpus uteri,Adenomas and Adenocarcinomas,976bb47a-d999-4e24-ad08-452fea73c846,"[{'primary_diagnosis': 'Clear cell carcinoma',...",TCGA-UCEC,clear_cell,unknown,Clear cell carcinoma,G1,NaN,...,corpus uteri,adenomas and adenocarcinomas,clear cell carcinoma,other,epithelial,uterine_clear_cell,ok,valid,1,0.001786
11427,Corpus uteri,Adenomas and Adenocarcinomas,e604f5d4-f2c0-4404-9073-b8dbc8396778,[{'primary_diagnosis': 'Endometrioid adenocarc...,TCGA-UCEC,endometrioid,unknown,"Endometrioid adenocarcinoma, NOS",G3,NaN,...,corpus uteri,adenomas and adenocarcinomas,endometrioid adenocarcinoma,adenocarcinoma,epithelial,uterine_endometrioid,ok,valid,1,0.001786


In [23]:
print(len(df_all_samples))
df_all_samples.tail(3)

245657


,case_id,submitter_id,sample_id,sample_type,barcode_id,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
245654,2fe46394-cfc5-44bb-81a9-e438a0126f5b,TCGA-A5-A1OK,65208656-d7e1-493a-8806-90b4e82d701f,Blood Derived Normal,TCGA-A5-A1OK-10A,c1d633f4-e539-4abd-9ba6-8c67d8b3f474,42f61f26-fbd1-4ecf-b446-980cdd7cdb8e.wxs.aliqu...,Aggregated Somatic Mutation,MAF,TCGA-UCEC,squamous,squamous_cell_carcinoma,squamous,unknown
245655,2fe46394-cfc5-44bb-81a9-e438a0126f5b,TCGA-A5-A1OK,65208656-d7e1-493a-8806-90b4e82d701f,Blood Derived Normal,TCGA-A5-A1OK-10A,37650872-d6af-4211-bd15-b7441ba6c378,0dbc85ff-ee09-467c-81ff-a6184401f7cc.wxs.Pinde...,Annotated Somatic Mutation,MAF,TCGA-UCEC,squamous,squamous_cell_carcinoma,squamous,unknown
245656,2fe46394-cfc5-44bb-81a9-e438a0126f5b,TCGA-A5-A1OK,65208656-d7e1-493a-8806-90b4e82d701f,Blood Derived Normal,TCGA-A5-A1OK-10A,cea3e10a-4c23-473a-8302-2e6a91a2b1dc,CABAL_p_TCGA_b112_125_SNP_N_GenomeWideSNP_6_F0...,Masked Copy Number Segment,TXT,TCGA-UCEC,squamous,squamous_cell_carcinoma,squamous,unknown


In [25]:
barcodes = np.unique(df_all_samples.barcode_id)
len(barcodes)

8506

In [26]:
sample_types = np.unique(df_all_samples.sample_type)
sample_types

array(['Additional - New Primary', 'Additional Metastatic',
       'Blood Derived Normal', 'Bone Marrow Normal', 'Buccal Cell Normal',
       'Metastatic', 'Primary Blood Derived Cancer - Peripheral Blood',
       'Primary Tumor', 'Recurrent Tumor', 'Solid Tissue Normal'],
      dtype=object)

In [27]:
df_normal = df_all_samples[df_all_samples.sample_type.str.contains('Normal', case=False, na=False)]
len(df_normal)

98079

In [32]:
symbols = np.unique(df_all_mutations.symbol)
len(symbols)

18961

In [29]:
primary_sites = list(np.unique(df_all_cases.primary_site_norm))
primary_sites.sort()
print(len(primary_sites))

print("\n".join(primary_sites))

57
adrenal gland
anus and anal canal
base of tongue
bladder
bones joints and articular cartilage of limbs
bones joints and articular cartilage of other and unspecified sites
brain
breast
bronchus and lung
cervix uteri
colon
connective subcutaneous and other soft tissues
corpus uteri
esophagus
eye and adnexa
floor of mouth
gum
heart mediastinum and pleura
hematopoietic and reticuloendothelial systems
hypopharynx
kidney
larynx
lip
liver and intrahepatic bile ducts
lymph nodes
meninges
nasal cavity and middle ear
oropharynx
other and ill-defined sites
other and ill-defined sites in lip oral cavity and pharynx
other and unspecified major salivary glands
other and unspecified male genital organs
other and unspecified parts of biliary tract
other and unspecified parts of mouth
other and unspecified parts of tongue
other endocrine glands and related structures
ovary
palate
pancreas
parotid gland
peripheral nerves and autonomic nervous system
prostate gland
rectosigmoid junction
rectum
retrope

In [34]:
root_summary = os.path.join(gdc.root_data, 'summary')
create_dir(root_summary)

'/home/flavio/uv/perturb_agent/data/tcga/summary'

In [37]:
pid = 'TCGA'

In [39]:
fname = f'{pid}_summ_cases.tsv'
pdwritecsv(df_all_cases, fname, root_summary)

fname = f'{pid}_summ_samples.tsv'
pdwritecsv(df_all_samples, fname, root_summary)

fname = f'{pid}_summ_mutations.tsv'
pdwritecsv(df_all_mutations, fname, root_summary)

True

In [42]:
stri = f"Interfacing GDC {pid} data, one gathered:"
print(stri)
stri = f"\t- {len(df_all_cases)} cases."
print(stri)
stri = f"\t- {len(df_all_samples)} samples."
print(stri)
stri = f"\t- {len(df_all_mutations)} annotated mutations."
print(stri)
stri = f"\t- {len(symbols)} different genes."
print(stri)


Interfacing GDC TCGA data, one gathered:
	- 11428 cases.
	- 245657 samples.
	- 480826 annotated mutations.
	- 18961 different genes.
